In [25]:
# train_model_classification.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. LOAD DATA
print("\n[1/7] Membaca data...")
data = pd.read_csv('student-por.csv', sep=';')
print(f"   Total data: {len(data)} siswa")


[1/7] Membaca data...
   Total data: 649 siswa


In [ ]:
# 2. BUAT TARGET KELULUSAN
print("\n[2/7] Membuat target kelulusan...")

# Standar kelulusan: nilai G3 >= 10 (skala 0-20)
data['lulus'] = (data['G3'] >= 10).astype(int)

jumlah_lulus = (data['lulus'] == 1).sum()
jumlah_tidak_lulus = (data['lulus'] == 0).sum()

print(f"   Siswa LULUS (G3 >= 10)     : {jumlah_lulus} siswa ({jumlah_lulus/len(data)*100:.1f}%)")
print(f"   Siswa TIDAK LULUS (G3 < 10): {jumlah_tidak_lulus} siswa ({jumlah_tidak_lulus/len(data)*100:.1f}%)")


[2/7] Membuat target kelulusan...
   Siswa LULUS (G3 >= 10)     : 549 siswa (84.6%)
   Siswa TIDAK LULUS (G3 < 10): 100 siswa (15.4%)


In [ ]:
# 3. PILIH FITUR (SEMUA NUMERIK)
print("\n[3/7] Memilih fitur...")

# fitur yang sudah pasti numerik atau akan diencode
fitur_pilihan = [
    'G1',           # Nilai periode 1 (numerik)
    'G2',           # Nilai periode 2 (numerik)
    'failures',     # Jumlah gagal kelas (numerik)
    'absences',     # Jumlah absensi (numerik)
    'studytime',    # Waktu belajar (numerik 1-4)
    'goout',        # Frekuensi hangout (numerik 1-5)
    'freetime',     # Waktu luang (numerik 1-5)
    'health',       # Status kesehatan (numerik 1-5)
    'higher',       # Ingin pendidikan tinggi (akan diencode)
    'internet'      # Akses internet (akan diencode)
]

X = data[fitur_pilihan].copy()
y = data['lulus']

print(f"   Fitur yang digunakan: {len(fitur_pilihan)} fitur")


[3/7] Memilih fitur...
   Fitur yang digunakan: 10 fitur


In [ ]:
# 4. KONVERSI 'yes'/'no' KE 1/0
print("\n[4/7] Mengubah 'yes'/'no' menjadi 1/0...")

# Konversi kolom higher
if 'higher' in X.columns:
    X['higher'] = X['higher'].map({'yes': 1, 'no': 0})
    print(f"   higher: 'yes' -> 1, 'no' -> 0")

# Konversi kolom internet
if 'internet' in X.columns:
    X['internet'] = X['internet'].map({'yes': 1, 'no': 0})
    print(f"   internet: 'yes' -> 1, 'no' -> 0")


[4/7] Mengubah 'yes'/'no' menjadi 1/0...
   higher: 'yes' -> 1, 'no' -> 0
   internet: 'yes' -> 1, 'no' -> 0


In [ ]:
# 5. CEK TIPE DATA
print("\n[5/7] Memastikan semua data numerik...")

print("\n   Tipe data setelah konversi:")
for col in X.columns:
    print(f"      {col}: {X[col].dtype}")

# Cek apakah masih ada yang object/string
if X.select_dtypes(include=['object']).shape[1] > 0:
    print("\n   MASIH ADA DATA STRING! Kolom:", X.select_dtypes(include=['object']).columns.tolist())
else:
    print("\n   Semua data sudah numerik!")

# Tampilkan sample
print("\n   Sample data (5 baris pertama):")
print(X.head())


[5/7] Memastikan semua data numerik...

   Tipe data setelah konversi:
      G1: int64
      G2: int64
      failures: int64
      absences: int64
      studytime: int64
      goout: int64
      freetime: int64
      health: int64
      higher: int64
      internet: int64

   Semua data sudah numerik!

   Sample data (5 baris pertama):
   G1  G2  failures  absences  studytime  goout  freetime  health  higher  \
0   0  11         0         4          2      4         3       3       1   
1   9  11         0         2          2      3         3       3       1   
2  12  13         0         6          2      2         3       3       1   
3  14  14         0         0          3      2         2       5       1   
4  11  13         0         0          2      2         3       5       1   

   internet  
0         0  
1         1  
2         1  
3         1  
4         0  


In [ ]:
# 6. BAGI DATA TRAIN DAN TEST
print("\n[6/7] Membagi data training dan testing...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"   Data training : {len(X_train)} siswa")
print(f"   Data testing  : {len(X_test)} siswa")


[6/7] Membagi data training dan testing...
   Data training : 519 siswa
   Data testing  : 130 siswa


In [ ]:
# 7. TRAINING GRADIENT BOOSTING
print("\n[7/7] Training Gradient Boosting...")

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

gb_model.fit(X_train, y_train)
print("   Model selesai dilatih!")


[7/7] Training Gradient Boosting...


   Model selesai dilatih!


In [ ]:

# 8. EVALUASI MODEL
print("\nEvaluasi model...")

# Prediksi
y_pred = gb_model.predict(X_test)
y_proba = gb_model.predict_proba(X_test)[:, 1]

# Metrik
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

# Cross validation
cv_scores = cross_val_score(gb_model, X_train, y_train, cv=5, scoring='accuracy')

print("\n   HASIL EVALUASI:")
print("   " + "-"*40)
print(f"   Accuracy  : {accuracy:.2%}")
print(f"   Precision : {precision:.2%}")
print(f"   Recall    : {recall:.2%}")
print(f"   F1-Score  : {f1:.2%}")
print(f"   AUC       : {auc:.2%}")
print(f"   CV Mean   : {cv_scores.mean():.2%} (+/- {cv_scores.std():.2%})")
print("   " + "-"*40)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print("\n   CONFUSION MATRIX:")
print("   " + "-"*55)
print(f"   {'':<20} {'Prediksi LULUS':<18} {'Prediksi TIDAK LULUS':<20}")
print("   " + "-"*55)
print(f"   {'Aktual LULUS':<20} {cm[1,1]:<18} {cm[1,0]:<20}")
print(f"   {'Aktual TIDAK LULUS':<20} {cm[0,1]:<18} {cm[0,0]:<20}")
print("   " + "-"*55)


Evaluasi model...

   HASIL EVALUASI:
   ----------------------------------------
   Accuracy  : 90.00%
   Precision : 93.69%
   Recall    : 94.55%
   F1-Score  : 94.12%
   AUC       : 94.05%
   CV Mean   : 91.72% (+/- 3.57%)
   ----------------------------------------

   CONFUSION MATRIX:
   -------------------------------------------------------
                        Prediksi LULUS     Prediksi TIDAK LULUS
   -------------------------------------------------------
   Aktual LULUS         104                6                   
   Aktual TIDAK LULUS   7                  13                  
   -------------------------------------------------------


In [ ]:
# 9. FEATURE IMPORTANCE
print("\n   FEATURE IMPORTANCE (pengaruh fitur):")
print("   " + "-"*50)

importance_df = pd.DataFrame({
    'Fitur': fitur_pilihan,
    'Penting': gb_model.feature_importances_
}).sort_values('Penting', ascending=False)

for i, row in importance_df.iterrows():
    bar = "█" * int(row['Penting'] * 50)
    print(f"   {row['Fitur']:<15} {bar} {row['Penting']:.2%}")
print("   " + "-"*50)


   FEATURE IMPORTANCE (pengaruh fitur):
   --------------------------------------------------
   G2              ████████████████████████████ 57.35%
   G1              ███████ 14.16%
   absences        ███ 7.51%
   goout           ██ 5.62%
   health          ██ 4.98%
   failures        ██ 4.59%
   studytime       █ 2.40%
   freetime        █ 2.01%
   internet         1.18%
   higher           0.20%
   --------------------------------------------------


In [ ]:
# 10. SIMPAN MODEL
print("\nMenyimpan model...")

# Buat folder jika belum ada
import os
os.makedirs('classification', exist_ok=True)

# Simpan model
joblib.dump(gb_model, 'classification/model_gradient_boosting.pkl')
print("   classification/model_gradient_boosting.pkl")

# Simpan fitur
joblib.dump(fitur_pilihan, 'classification/fitur_kelulusan.pkl')
print("   classification/fitur_kelulusan.pkl")

# Simpan mapping untuk higher dan internet (karena kita tidak pakai encoder)
mapping = {
    'higher': {'yes': 1, 'no': 0},
    'internet': {'yes': 1, 'no': 0}
}
joblib.dump(mapping, 'classification/mapping_kelulusan.pkl')
print("   classification/mapping_kelulusan.pkl")

# Simpan hasil evaluasi
hasil = {
    'model': 'Gradient Boosting',
    'akurasi': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'auc': float(auc),
    'fitur': fitur_pilihan,
    'confusion_matrix': cm.tolist(),
    'jumlah_siswa': {
        'total': len(data),
        'lulus': int(jumlah_lulus),
        'tidak_lulus': int(jumlah_tidak_lulus)
    }
}

import json
def convert(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return str(obj)

with open('classification/hasil_evaluasi.json', 'w') as f:
    json.dump(hasil, f, indent=2, default=convert)
print("   classification/hasil_evaluasi.json")


Menyimpan model...
   classification/model_gradient_boosting.pkl
   classification/fitur_kelulusan.pkl
   classification/mapping_kelulusan.pkl
   classification/hasil_evaluasi.json


In [36]:
# 11. RINGKASAN
print("\n" + "="*70)
print(" " * 20 + "MODEL CLASSIFICATION SELESAI DISIMPAN!")
print("="*70)

print("\nStruktur folder:")
print("   classification/")
print("   ├── model_gradient_boosting.pkl")
print("   ├── fitur_kelulusan.pkl")
print("   ├── mapping_kelulusan.pkl")
print("   └── hasil_evaluasi.json")

print("\nRingkasan Akhir:")
print(f"   Model: Gradient Boosting Classifier")
print(f"   Target: LULUS (G3 ≥ 10) atau TIDAK LULUS")
print(f"   Akurasi: {accuracy:.2%}")
print(f"   AUC: {auc:.2%}")

print("\n10 Fitur yang digunakan:")
for i, f in enumerate(fitur_pilihan, 1):
    print(f"   {i}. {f}")

print("\n" + "="*70)


                    MODEL CLASSIFICATION SELESAI DISIMPAN!

Struktur folder:
   classification/
   ├── model_gradient_boosting.pkl
   ├── fitur_kelulusan.pkl
   ├── mapping_kelulusan.pkl
   └── hasil_evaluasi.json

Ringkasan Akhir:
   Model: Gradient Boosting Classifier
   Target: LULUS (G3 ≥ 10) atau TIDAK LULUS
   Akurasi: 90.00%
   AUC: 94.05%

10 Fitur yang digunakan:
   1. G1
   2. G2
   3. failures
   4. absences
   5. studytime
   6. goout
   7. freetime
   8. health
   9. higher
   10. internet

